# Parse Email

Parse mbox-format `.txt` files in `sample-data/` into individual emails.

Each file is a thread: messages separated by mbox `From ` envelope lines, each a standard RFC-822 message (`From:`, `To:`, `Date:`, `Subject:` headers + body).


In [ ]:
import mailbox
import re
from email.utils import getaddresses
from pathlib import Path

SAMPLE_DIR = Path("..") / "sample-data"

In [ ]:
def addrs(header):
    """Extract sorted, lowercased email addresses from a To/From header."""
    if not header:
        return []
    return sorted(a.lower() for _, a in getaddresses([header]) if a)


def normalize(text):
    """Collapse whitespace so near-duplicates compare equal."""
    return re.sub(r"\s+", " ", text).strip()


def parse_file(path):
    """Parse one mbox .txt thread into a list of email dicts."""
    emails = []
    for order, msg in enumerate(mailbox.mbox(str(path))):
        body = msg.get_payload()
        emails.append(
            {
                "doc_id": path.name,
                "canon_order": order,
                "from": addrs(msg["From"]),
                "to": addrs(msg["To"]),
                "subject": msg["Subject"],
                "date": msg["Date"],
                "content": normalize(body),
            }
        )
    return emails

In [ ]:
all_emails = []
for path in sorted(SAMPLE_DIR.glob("*.txt")):
    emails = parse_file(path)
    all_emails.extend(emails)
    print(f"{path.name}: {len(emails)} emails")

print(f"\ntotal: {len(all_emails)} emails")

In [ ]:
for e in all_emails:
    print(f"[{e['doc_id']} #{e['canon_order']}] {e['from']} -> {e['to']}")
    print(f"  subject: {e['subject']}")
    print(f"  content: {e['content']}")
    print()

In [ ]:
all_emails

## Rapidfuzz similarity

Try `rapidfuzz` Levenshtein on the parsed email contents. The emails-consumer uses
`Levenshtein.normalized_similarity > 0.9` to flag near-duplicates (after an exact
`content_hash` check). Below we score every pair sharing the same `from`/`to` to see
where the near-dup pairs land.


In [ ]:
!pip install rapidfuzz

In [ ]:
from rapidfuzz.distance import Levenshtein

# the two known near-duplicate pairs from the sample data
pairs = [
    (
        "doc2#0 vs doc3#0 (whitespace)",
        "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice",
        "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice",
    ),
    (
        "doc4#2 vs doc5#2 (one word inserted)",
        "Perfect, 10am Tuesday it is. I'll send a calendar invite. Alice",
        "Perfect, 10am on Tuesday it is. I'll send a calendar invite. Alice",
    ),
    (
        "kickoff vs reply (different email)",
        "Hi Bob, Let's schedule the project kickoff for next week. Does Tuesday work for you? Alice",
        "Hi Alice, Tuesday works great for me. Let's say 10am? Bob",
    ),
]

for label, a, b in pairs:
    print(f"{Levenshtein.normalized_similarity(a, b):.3f}  {label}")